<a href="https://colab.research.google.com/github/NguyenBaTam-tristan/deepml-curriculum-scraper/blob/main/Designed_curriculum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#!/usr/bin/env python3
"""
Sinh lịch học Deep-ML System/Edge (07/10/2026 -> 31/12/2026).
Tập trung: Karpathy Style (From scratch, Autograd, Systems, Memory, Inference).
Cắt bỏ: Thống kê cổ điển, Kinh tế lượng, Data Science truyền thống.

Chạy lệnh:  python build_schedule_v2.py
Cần có file: deepml_v2_20261006_043044.json
"""
import csv, json, re, sys
from datetime import date, timedelta

SRC = "deepml_v2_20261006_043044.json"
OUT = "learning_schedule_system_edge_2026.csv"
START, END = date(2026, 10, 7), date(2026, 12, 31)

# ---------- ước lượng thời gian (giờ / bài) ----------
H = {"Easy": 0.6, "Medium": 1.2, "Hard": 2.5, "math": 0.5}
DAY_MIN, DAY_MAX = 3.5, 4.0
RANK = {"Easy": 0, "Medium": 1, "Hard": 2}

# ---------- 51 bài đã làm (khớp theo tên) ----------
DONE_PROBLEMS = """Calculate 2x2 Matrix Inverse|Calculate Accuracy Score|Calculate Cosine Similarity Between Vectors|
Calculate F1 Score from Predicted and True Labels|Calculate Mean by Row or Column|Compute the Cross Product of Two 3D Vectors|
Convert Vector to Diagonal Matrix|Derivative of a Polynomial|Descriptive Statistics Calculator|Dot Product Calculator|
Empirical Probability Mass Function (PMF)|Expected Value and Variance of an n-Sided Die|Feature Scaling Implementation|
Gradient Direction and Magnitude|Linear Regression Using Gradient Descent|Linear Regression Using Normal Equation|
Matrix Determinant & Trace|Matrix-Vector Dot Product|Min-Max Scaling of Feature Values|One-Hot Encoding of Nominal Values|
Reshape Matrix|Scalar Multiplication of a Matrix|Sigmoid Activation Function Understanding|Softmax Activation Function Implementation|
Transformation Matrix from Basis B to C|Transpose of a Matrix|Vector Element-wise Sum|Vector Norms (L1/L2/L-inf) and the Frobenius Norm|
Calculate Eigenvalues of a Matrix|Implement K-Fold Cross-Validation|Implement K-Nearest Neighbors|Matrix times Matrix|Matrix Transformation|
Quotient Rule for Derivatives""".replace("\n", "").split("|")
DONE_MATH = """Derivatives and Gradients|Descriptive Statistics|Expectation and Variance Algebra|Gradient Descent Updates|Matrix Basics|
ML Workflow Basics|Probability Fundamentals|Vector Operations|Covariance and Correlation|Determinants and Trace|Inverse and Rank|
Least Squares and the Normal Equations|Matrix Multiplication|Orthogonality and Projections|Solving Linear Systems|
The Four Fundamental Subspaces|Vector Norms and Linear Independence""".replace("\n", "").split("|")

norm = lambda s: re.sub(r"\s+", " ", s.strip().lower())

# ---------- 8 Chủ đề cốt lõi cho System/Edge Engineer ----------
THEMES = {
    1: ("Linear Algebra & Calculus", "Toán lõi (SVD, Eigen, Jacobian, Hessian) phục vụ trực tiếp cho LoRA, Gradient và PCA."),
    2: ("Information Theory & Loss", "Hiểu bản chất Entropy, KL Divergence, MLE để phân tích Perplexity và Cross-Entropy lúc nén LLM."),
    3: ("Autograd & Custom Engines", "Tự viết engine (micrograd/teenygrad style), code forward/backward pass và optimizer từ đầu."),
    4: ("Transformer Architecture", "Xây dựng các khối Attention (MHA/GQA/MLA), RoPE, Normalization (nanoGPT style)."),
    5: ("LLM Inference & Mechanics", "Tối ưu KV-Cache, PagedAttention, Speculative Decoding, Roofline model, phân tích Bottleneck."),
    6: ("Quantization & Edge AI", "Kỹ thuật ép kiểu FP16/BF16/INT8/FP8/FP4, nén mô hình và triển khai trên Edge/Local."),
    7: ("Systems, Kernels & Parallelism", "Tư duy hệ thống: DDP, Ring All-Reduce, Memory Hierarchy, Kernel fusion, Tensor Parallelism."),
    8: ("RLHF & Alignment (Core)", "Hiểu cơ chế tinh chỉnh DPO/PPO/GRPO căn bản để căn chỉnh mô hình nhỏ chạy Edge.")
}

# math: Luật ưu tiên giữ lại các bài toán lõi, bỏ qua thống kê phức tạp
MATH_RULES = [
    (1, r"^matrix calculus|^eigen|^matrix decomp|^svd|^positive definite|^multivariate calculus|^taylor|^hessian|^jacobian|^derivatives|^least squares|^orthogonal"),
    (2, r"probab|bayes|gaussian|kl divergence|entropy|likelihood|information|cross-entropy"),
]

# problems: Tập trung săn các từ khóa Hệ thống và C/CUDA
CORE_KW = r"autograd|backprop|backward|adam|muon|optimizer|attention|mqa|gqa|mla|paged|flash|rope|rotary|kv.?cache|quantiz|int8|fp8|fp4|mxfp4|continuous batching|roofline|speculative|tokeniz|bpe|lora|moe|mixture of experts|dpo|ppo|grpo|all-reduce|ddp|parallel|latency|throughput|bandwidth|fusion|kernel|memory|gpt|transformer|cuda|triton|cpp"

# DANH SÁCH ĐEN: Chặn tuyệt đối thống kê sinh học, kinh tế lượng, metrics cũ, và game RL cổ điển
NEG_KW = r"anova|chi-square|t-test|hypothesis|p-value|confidence interval|meteor|bleu|rouge|ab test|canary|kaplan-meier|survival|cox|instrumental variable|regression discontinuity|simpson|cuped|treatment|blackjack|cliff walking|gridworld|taxi|mountain car|atari|asr|tts|audio|speech"

MANDATORY = {"Inference", "Optimization", "MLOps", "Performance Optimization", "Systems Programming"}
EXTRA_CATS = {"Deep Learning", "pytorch", "cuda", "triton", "Compilers", "tinygrad", "LLM", "Linear Algebra", "Machine Learning Systems"}

P_RULES = [
    (7, r"cuda|triton|compiler|mlops|kernel|parallel|all-reduce|ddp|mfu|autoscal|serving|deploy|pipeline|tinygrad|tensor core|profil|performance|communication cost|nvlink|mig|memory hierarchy"),
    (6, r"quantiz|int8|fp8|fp4|mxfp4|lora|moe|mixture of experts|pruning|distillation|sparsity|mini-float|mixed precision|number format"),
    (5, r"kv.?cache|paged|flash|continuous batching|speculative|prefill|decode|decoding|sampling|roofline|arithmetic intensity|tiled attention|latency|throughput|bandwidth|fragmentation"),
    (4, r"attention|mqa|gqa|mla|rope|rotary|positional|transformer|causal|mask|gpt"),
    (3, r"autograd|backprop|backward|gradient|adam|sgd|muon|momentum|optimizer|scheduler|loss|cross-entropy|softmax|hessian|jacobian"),
    (8, r"dpo|ppo|grpo|rlhf|reward|rloo|gae|preference"),
    (2, r"entropy|kl |likelihood|information theory"),
    (1, r"matrix|eigen|svd|determinant|inverse|decomposition|qr|lu |orthogonal|tensor"),
]

def theme_of(title, rules, default):
    t = title.lower()
    for th, pat in rules:
        if re.search(pat, t):
            return th
    return default

def prio(x):
    t, c = x["title"].lower(), x["category"]
    s = 0.0
    if c in MANDATORY: s += 6
    if re.search(CORE_KW, t): s += 5
    if c in ("Machine Learning Systems", "Inference"): s += 3
    s += {"Easy": 0, "Medium": 1.0, "Hard": 0.5}[x["difficulty"]]
    if re.search(NEG_KW, t): s -= 20 # Phạt cực nặng để loại bỏ bài không liên quan
    return s

def load():
    try:
        with open(SRC, encoding="utf-8") as f: d = json.load(f)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file {SRC}")
        sys.exit(1)

    done_p = {norm(t) for t in DONE_PROBLEMS}
    done_m = {norm(t) for t in DONE_MATH}
    math = [x for x in d if x.get("section") == "math"]
    probs = [x for x in d if x.get("section") == "problems"]
    labs = {x["id"]: x for x in d if x.get("section") == "labs"}
    projs = {x["id"]: x for x in d if x.get("section") == "projects"}
    return math, probs, done_p, done_m, labs, projs

def build_pools(math, probs, done_p, done_m):
    M = []
    for i, x in enumerate(math):
        if norm(x["title"]) in done_m or re.search(NEG_KW, x["title"].lower()):
            continue
        th = theme_of(x["title"], MATH_RULES, 0)
        if th != 0: # Chỉ lấy math thuộc theme 1 và 2
            M.append(dict(x, theme=th, order=-i, h=H["math"], diff="Medium"))

    seen, cand = set(), []
    for x in probs:
        k, t, c = norm(x["title"]), x["title"].lower(), x.get("category", "")
        if k in done_p or k in seen or re.search(NEG_KW, t):
            continue
        # Bộ lọc chặt chẽ hơn:
        if c in MANDATORY or c in EXTRA_CATS or re.search(CORE_KW, t):
            seen.add(k)
            cand.append(dict(x, theme=theme_of(x["title"], P_RULES, 7 if c in MANDATORY else 4),
                             order=int(x["id"]), h=H[x["difficulty"]], diff=x["difficulty"], pr=prio(x)))
    cand.sort(key=lambda x: (-x["pr"], x["order"]))
    return M, cand

# Giữ nguyên cấu trúc Project xuất sắc của Karpathy
LAB_BY_SAT = {
    date(2026, 10, 10): "design-your-own-optimizer-numpy",
    date(2026, 10, 17): "design-your-own-attention-mechanism",
    date(2026, 11, 7): "design-your-own-kv-cache-eviction-policy",
    date(2026, 11, 21): "design-your-own-quantizer",
    date(2026, 12, 5): "design-your-own-request-scheduler",
}
PROJ_BY_SAT = {
    date(2026, 10, 31): "build-your-own-teenygrad-a-tiny-tensor-autograd-engine",
    date(2026, 11, 14): "tiny-gpt-from-scratch",
    date(2026, 11, 28): "llm-inference-mechanics-prefill-decode-and-the-kv-cache",
    date(2026, 12, 12): "on-device-model-compression-pipeline",
    date(2026, 12, 26): "build-a-mini-llm-inference-server",
}
PROJ_FRI = {s - timedelta(days=1): (s, p) for s, p in PROJ_BY_SAT.items()}

# Ánh xạ ID project/lab sang Theme mới
LAB_THEME = {"design-your-own-optimizer-numpy": 3, "design-your-own-attention-mechanism": 4,
             "design-your-own-kv-cache-eviction-policy": 5, "design-your-own-quantizer": 6,
             "design-your-own-request-scheduler": 7}
PROJ_THEME = {"build-your-own-teenygrad-a-tiny-tensor-autograd-engine": 3, "tiny-gpt-from-scratch": 4,
              "llm-inference-mechanics-prefill-decode-and-the-kv-cache": 5, "on-device-model-compression-pipeline": 6,
              "build-a-mini-llm-inference-server": 7}

def weekday_slots():
    d, out = START, []
    while d <= END:
        if d.weekday() <= 4 and d not in PROJ_FRI:
            out.append(d)
        d += timedelta(days=1)
    return out

def pack(M, P, slots):
    M = sorted(M, key=lambda x: (x["theme"], x["order"]))
    P = sorted(P, key=lambda x: (x["theme"], RANK[x["diff"]], x["order"]))
    days = []
    for d in slots:
        if not M and not P: break
        cur = min([x["theme"] for x in (M[:1] + P[:1])])
        mi, pi, hrs = [], [], 0.0
        hard_idx = next((i for i, p in enumerate(P) if p["diff"] == "Hard" and p["theme" ] <= cur), None)
        non_hard_left = any(p["diff"] != "Hard" and p["theme"] <= cur for p in P)

        if hard_idx is not None and not non_hard_left:
            hp = P.pop(hard_idx)
            pi.append(hp); hrs += hp["h"]
            mid = next((i for i, p in enumerate(P) if p["diff"] == "Medium"), None)
            if mid is not None:
                q = P.pop(mid); pi.append(q); hrs += q["h"]
        else:
            while P and P[0]["diff"] != "Hard" and hrs + P[0]["h"] <= 2.5 and len(pi) < 4:
                q = P.pop(0); pi.append(q); hrs += q["h"]
            while M and hrs + M[0]["h"] <= DAY_MAX and (hrs < DAY_MIN or len(mi) < 2) and len(mi) < 8:
                q = M.pop(0); mi.append(q); hrs += q["h"]
            while P and P[0]["diff"] != "Hard" and hrs + P[0]["h"] <= DAY_MAX and hrs < DAY_MIN:
                q = P.pop(0); pi.append(q); hrs += q["h"]
            while M and hrs + M[0]["h"] <= DAY_MAX and hrs < DAY_MIN:
                q = M.pop(0); mi.append(q); hrs += q["h"]
        days.append((d, mi, pi, round(hrs, 2)))
    return days, M, P

def fit_problems(M, cand, slots):
    lo, hi, best = 0, len(cand), 0
    while lo <= hi:
        mid = (lo + hi) // 2
        _, rm, rp = pack([dict(x) for x in M], [dict(x) for x in cand[:mid]], slots)
        if not rm and not rp:
            best, lo = mid, mid + 1
        else:
            hi = mid - 1
    return best

def overall(items):
    r = [{"Easy": 1, "Medium": 2, "Hard": 3}[i["diff"]] for i in items]
    m = sum(r) / len(r) if r else 1
    return "Easy" if m < 1.5 else ("Medium" if m < 2.4 else "Hard")

def main():
    math, probs, done_p, done_m, labs, projs = load()
    M, cand = build_pools(math, probs, done_p, done_m)
    slots = weekday_slots()
    n = fit_problems(M, cand, slots)
    chosen = cand[:n]
    days, _, _ = pack([dict(x) for x in M], [dict(x) for x in chosen], slots)

    print(f"Toán lõi đã lọc: {len(M)} bài | Problems System/Edge ứng viên: {len(cand)} bài")
    print(f"Đã xếp: {n} problems vào {len(days)}/{len(slots)} ngày thường.")

    rows, week_probs = {}, {}
    for d, mi, pi, hrs in days:
        items = mi + pi
        th = [x["theme"] for x in items]
        main_t = max(set(th), key=th.count) if th else 7
        sec = ";".join(s for s, l in (("math", mi), ("problems", pi)) if l)
        rows[d] = dict(Section=sec, Item_ID=";".join(x["id"] for x in items),
                       Task_Title="; ".join(x["title"].strip() for x in items),
                       Category=THEMES[main_t][0], Difficulty=overall(items),
                       Estimated_Hours=f"{hrs:.2f}".rstrip("0").rstrip("."), Focus_Note=THEMES[main_t][1])
        week_probs.setdefault(d.isocalendar()[1], []).extend(pi)

    def review_for(sat):
        wk = sat.isocalendar()[1]
        pool = sorted(week_probs.get(wk, []), key=lambda x: -RANK[x["diff"]])[:2]
        return pool

    for d in [START + timedelta(days=i) for i in range((END - START).days + 1)]:
        if d in PROJ_FRI:
            s, pid = PROJ_FRI[d]
            if pid in projs:
                p = projs[pid]
                rows[d] = dict(Section="projects", Item_ID=pid, Task_Title=f"{p['title']} (Part 1/2: dựng khung & core)",
                               Category=THEMES[PROJ_THEME[pid]][0], Difficulty="Hard", Estimated_Hours="3.75",
                               Focus_Note="Tự tay dựng thành phần cốt lõi from scratch - " + THEMES[PROJ_THEME[pid]][1])
        elif d in PROJ_BY_SAT:
            pid = PROJ_BY_SAT[d]
            if pid in projs:
                p = projs[pid]; rv = review_for(d)
                rows[d] = dict(Section="projects", Item_ID=";".join([pid] + [x["id"] for x in rv]),
                               Task_Title="; ".join([f"{p['title']} (Part 2/2: hoàn thiện & test)"] + [f"[Review] {x['title'].strip()}" for x in rv]),
                               Category=THEMES[PROJ_THEME[pid]][0], Difficulty="Hard", Estimated_Hours="3.75",
                               Focus_Note="Hoàn tất project + review code hệ thống - " + THEMES[PROJ_THEME[pid]][1])
        elif d in LAB_BY_SAT:
            lid = LAB_BY_SAT[d]
            if lid in labs:
                l = labs[lid]; rv = review_for(d)
                rows[d] = dict(Section="labs", Item_ID=";".join([lid] + [x["id"] for x in rv]),
                               Task_Title="; ".join([l["title"]] + [f"[Review] {x['title'].strip()}" for x in rv]),
                               Category=THEMES[LAB_THEME[lid]][0], Difficulty="Medium", Estimated_Hours="3.5",
                               Focus_Note="Lab tổng hợp cuối tuần + review code lõi - " + THEMES[LAB_THEME[lid]][1])
        elif d.weekday() == 6:
            rows[d] = dict(Section="", Item_ID="", Task_Title="Rest Day - nghỉ hoàn toàn", Category="Rest",
                           Difficulty="", Estimated_Hours="0", Focus_Note="Nghỉ ngơi để não củng cố kiến trúc System vừa nạp.")
        elif d not in rows:
            rows[d] = dict(Section="", Item_ID="", Task_Title="Buffer / Thử nghiệm Kernel C++ or CUDA",
                           Category="Review & Hack", Difficulty="Medium", Estimated_Hours="3.5",
                           Focus_Note="Ngày đệm: Dùng để đọc source code llama.cpp, micrograd, hoặc viết Triton kernel.")

    cols = ["Date", "DayOfWeek", "Type", "Section", "Item_ID", "Task_Title", "Category", "Difficulty", "Estimated_Hours", "Focus_Note"]
    with open(OUT, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        d = START
        while d <= END:
            r = rows.get(d, {})
            if r:
                w.writerow(dict(Date=d.isoformat(), DayOfWeek=d.strftime("%A"), Type="Rest" if d.weekday() == 6 else "Study", **r))
            d += timedelta(days=1)

    print(f"Đã ghi thành công lộ trình System/Edge vào: {OUT}")

if __name__ == "__main__":
    main()

Toán lõi đã lọc: 31 bài | Problems System/Edge ứng viên: 704 bài
Đã xếp: 155 problems vào 57/57 ngày thường.
Đã ghi thành công lộ trình System/Edge vào: learning_schedule_system_edge_2026.csv
